<a href="https://colab.research.google.com/github/anbuchezhiyan2005/Image-Captioning-System/blob/main/notebooks/Image_Captioning_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!git clone https://github.com/anbuchezhiyan2005/Image-Captioning-System.git

Cloning into 'Image-Captioning-System'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 136 (delta 66), reused 114 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 47.43 KiB | 1.90 MiB/s, done.
Resolving deltas: 100% (66/66), done.


In [6]:
!cd /content/Image-Captioning-System && git pull

Already up to date.


In [7]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

PyTorch version: 2.10.0+cpu
CUDA available: False
GPU device: No GPU


In [8]:
!pip install -q torch torchvision pillow

In [9]:
import torch
import torchvision
from PIL import Image
import json
import os
import sys

print("All packages imported successfully!")

All packages imported successfully!


In [10]:
DATASET_PATH = "/content/drive/MyDrive/Image-Captioning-System/dataset"

print("Checking dataset files...")
print(f"Dataset path: {DATASET_PATH}")
print(f"Path exists: {os.path.exists(DATASET_PATH)}")

if os.path.exists(DATASET_PATH):
  print("\n Dataset files in the folder: ")
  for item in os.listdir(DATASET_PATH):
    print(item)

  captions_file = os.path.join(DATASET_PATH, "captions_encoded.json")
  vocab_file = os.path.join(DATASET_PATH, "vocabulary.json")
  images_directory = os.path.join(DATASET_PATH, "Processed_Images")

  print(f"\n✅ captions_encoded.json exists: {os.path.exists(captions_file)}")
  print(f"✅ vocabulary.json exists: {os.path.exists(vocab_file)}")
  print(f"✅ Processed_Images/ exists: {os.path.exists(images_directory)}")

  if os.path.exists(images_directory):
      num_images = len(os.listdir(images_directory))
      print(f"✅ Number of images: {num_images}")
  else:
      print("❌ Dataset path not found!")
      print("Please upload your dataset to Google Drive first.")


Checking dataset files...
Dataset path: /content/drive/MyDrive/Image-Captioning-System/dataset
Path exists: True

 Dataset files in the folder: 
vocabulary.json
captions_encoded.json
Processed_Images

✅ captions_encoded.json exists: True
✅ vocabulary.json exists: True
✅ Processed_Images/ exists: True
✅ Number of images: 8091


In [11]:
PROJECT_ROOT = "/content/Image-Captioning-System"
sys.path.insert(0, PROJECT_ROOT)

DATASET_PATH = "/content/drive/MyDrive/Image-Captioning-System/dataset"

CAPTIONS_FILE = os.path.join(DATASET_PATH, "captions_encoded.json")
VOCAB_FILE = os.path.join(DATASET_PATH, "vocabulary.json")
IMAGE_DIR = os.path.join(DATASET_PATH, "Processed_Images")

print(f"\n✅ captions_encoded.json exists: {os.path.exists(CAPTIONS_FILE)}")
print(f"✅ vocabulary.json exists: {os.path.exists(VOCAB_FILE)}")
print(f"✅ Processed_Images/ exists: {os.path.exists(IMAGE_DIR)}")


✅ captions_encoded.json exists: True
✅ vocabulary.json exists: True
✅ Processed_Images/ exists: True


In [12]:
from models.encoder import CNNEncoder
from models.decoder import LSTMDecoder
from training.dataset import MyDataset, collate_fn
from training.train_config import get_loss_function, get_optimizer

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

with open(VOCAB_FILE, mode = "r") as f:
  vocab_data = json.load(f)
  VOCAB_SIZE = len(vocab_data)
  print(f"✅ Vocabulary size: {VOCAB_SIZE}")

✅ Vocabulary size: 8832


In [13]:
print("Creating dataset...")
dataset = MyDataset(CAPTIONS_FILE, IMAGE_DIR)
print(f"Created Dataset: {len(dataset)} samples")

# Creating DaaLoader
BATCH_SIZE = 32
NUM_WORKERS = 2

dataloader = DataLoader(
    dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collate_fn,
    num_workers = NUM_WORKERS,
    pin_memory =True
)

print(f"✅ DataLoader created: {len(dataloader)} batches")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Total samples: {len(dataset)}")

# Testing one batch
images, captions = next(iter(dataloader))
print(f"✅ Batch images shape: {images.shape}")
print(f"✅ Batch captions shape: {captions.shape}")

Creating dataset...
Created Dataset: 40455 samples
✅ DataLoader created: 1265 batches
   Batch size: 32
   Total samples: 40455


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✅ Batch images shape: torch.Size([32, 3, 224, 224])
✅ Batch captions shape: torch.Size([32, 21])


In [14]:
# Defining Hyperparameters
EMBED_SIZE = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 1
LEARNING_RATE = 0.001

# Device Configuration
device = torch.device('cuda'if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# Initialize models
print("Initializing models...")
encoder = CNNEncoder().to(device)
decoder = LSTMDecoder(
    embedding_dim = EMBED_SIZE,
    hidden_size = HIDDEN_SIZE,
    vocab_size = VOCAB_SIZE,
    num_layers = NUM_LAYERS
).to(device)

print(f"✅ Encoder initialized (embed_size={EMBED_SIZE})")
print(f"✅ Decoder initialized (hidden={HIDDEN_SIZE}, layers={NUM_LAYERS}, vocab={VOCAB_SIZE})")

# Count parameters
encoder_params = sum(p.numel() for p in encoder.parameters())
decoder_params = sum(p.numel() for p in decoder.parameters())
total_params = encoder_params + decoder_params

print(f"\n📊 Model Parameters:")
print(f"   Encoder: {encoder_params:,}")
print(f"   Decoder: {decoder_params:,}")
print(f"   Total: {total_params:,}")


✅ Device: cpu
Initializing models...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 140MB/s]


✅ Encoder initialized (embed_size=256)
✅ Decoder initialized (hidden=512, layers=1, vocab=8832)

📊 Model Parameters:
   Encoder: 23,508,032
   Decoder: 9,417,856
   Total: 32,925,888


In [15]:
# Loss Function
criterion = get_loss_function()
print(f"✅ Loss function: {criterion}")

params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = get_optimizer(params, LEARNING_RATE)

print(f"✅ Loss function: CrossEntropyLoss (ignore_index=0)")
print(f"✅ Optimizer: Adam (lr={LEARNING_RATE})")
print(f"✅ Total trainable parameters: {sum(p.numel() for p in params if p.requires_grad):,}")

✅ Loss function: CrossEntropyLoss()
✅ Loss function: CrossEntropyLoss (ignore_index=0)
✅ Optimizer: Adam (lr=0.001)
✅ Total trainable parameters: 32,925,888


In [41]:
def train_epoch(
    encoder,
    decoder,
    dataloader,
    criterion,
    optimizer,
    device
):

  # Setting models to training mode
  encoder.train()
  decoder.train()

  total_loss = 0
  num_batches = len(dataloader)

  # Implementing progress bar(tqdm)
  pbar = tqdm(dataloader, desc = "Training", unit = "batches")

  for (images, captions) in pbar:

    # moving images and captions to GPU
    images = images.to(device)
    captions = captions.to(device)

    # Setting gradients to zero
    optimizer.zero_grad()

    # Forward pass - getting features from the encoder
    features = encoder(images)

    # Forward pass - getting the outputs from the decoder
    outputs = decoder(features, captions[:, : -1])
    targets = captions[:, 1:]

    # reshaping outputs and captions for loss calculation
    outputs = outputs.view(-1, outputs.size(-1))
    targets = targets.view(-1)

    # Calculate the loss
    loss = criterion(outputs, targets)

    # Backward pass
    loss.backward()

    # gradient clipping (prevents exploding gradients)
    torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm = 5.0)
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm = 5.0)

    # Updating weights
    optimizer.step()

    # Updating total loss
    total_loss += loss.item()

    # tracking the loss
    pbar.set_postfix({'loss': f'{loss.item():.4f}'})

  avg_loss = total_loss / num_batches
  return avg_loss

print("Training loop defined!")






Training loop defined!


In [42]:
def validate(
    encoder,
    decoder,
    dataloader,
    criterion,
    device
):

  # Switching to evaluation mode
  encoder.eval()
  decoder.eval()

  total_loss = 0
  num_batches = len(dataloader)

  # Disabling gradient computation
  with torch.no_grad():

    pbar = tqdm(dataloader, desc = "validation", unit = "batches")

    for (images, captions) in pbar:

      # Moving images and captions to GPU
      images = images.to(device)
      captions = captions.to(device)

      # Forward pass (encoder + decoder)
      features = encoder(images)
      outputs = decoder(features, captions[:, : -1])
      target = captions[:, 1:]

      # reshaping outputs and captions for loss calculation
      outputs = outputs.view(-1, outputs.size(-1))
      target = target.view(-1)

      # Calculating the loss
      loss = criterion(outputs, target)

      # Tracking loss
      total_loss += loss.item()
      pbar.set_postfix({'loss': f'{loss.item():.4f}'})

  avg_loss = total_loss / num_batches
  return avg_loss

print("Validation function defined!")

Validation function defined!


In [18]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

print(f"Total samples: {len(dataset)}")
print(f"Train: {train_size} ({train_size/len(dataset)*100:.1f}%)")
print(f"Val: {val_size} ({val_size/len(dataset)*100:.1f}%)")
print(f"Test: {test_size} ({test_size/len(dataset)*100:.1f}%) \n")

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

print(f"✅ Datasets created!")
print(f"   Train: {len(train_dataset)} samples")
print(f"   Val: {len(val_dataset)} samples")
print(f"   Test: {len(test_dataset)} samples")

Total samples: 40455
Train: 32364 (80.0%)
Val: 4045 (10.0%)
Test: 4046 (10.0%) 

✅ Datasets created!
   Train: 32364 samples
   Val: 4045 samples
   Test: 4046 samples


In [19]:
# Creating training DataLoader
training_dataloader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collate_fn,
    num_workers = NUM_WORKERS,
    pin_memory = True
)

# Creating validation DataLoader
validation_dataloader = DataLoader(
    val_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collate_fn,
    num_workers = NUM_WORKERS,
    pin_memory = True
)

# Creating testing DataLoader
testing_dataloader = DataLoader(
    test_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collate_fn,
    num_workers = NUM_WORKERS,
    pin_memory = True
)

print(f"\n✅ DataLoaders created!")
print(f"   Train batches: {len(training_dataloader)}")
print(f"   Val batches: {len(validation_dataloader)}")
print(f"   Test batches: {len(testing_dataloader)}")


✅ DataLoaders created!
   Train batches: 1012
   Val batches: 127
   Test batches: 127


In [ ]:
# Training Configuration
NUM_EPOCHS = 10
SAVE_DIR = "/content/drive/MyDrive/Image-Captioning-System/checkpoints"

# Creating the checkpoint directory
os.makedirs(SAVE_DIR, exist_ok = True)

train_losses = []
validation_losses = []
best_validation_loss = float('inf')

print(f"Starting training for {NUM_EPOCHS} epochs...")
print(f"Checkpoints will be saved to: {SAVE_DIR}")
print("="*60)

for epoch in range(NUM_EPOCHS):
  print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
  print("-" * 40)

  train_loss = train_epoch(
      encoder,
      decoder,
      training_dataloader,
      criterion,
      optimizer,
      device
  )

  validation_loss = validate(
      encoder,
      decoder,
      training_dataloader,
      criterion,
      device
  )

  train_losses.append(train_loss)
  validation_losses.append(validation_loss)

  print(f"\nEpoch {epoch+1} Results:")
  print(f"  Train Loss: {train_loss:.4f}")
  print(f"  Val Loss: {validation_loss:.4f}")

  if validation_loss < best_validation_loss:
    best_validation_loss = validation_loss

    checkpoint = {
        'epoch': epoch + 1,
        'encoder_state_dict': encoder.state_dict(),
        'decoder_state_dict': decoder.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': validation_loss
    }

    checkpoint_path = os.path.join(SAVE_DIR, f'best_model_epoch_{epoch + 1}.pth')
    torch.save(checkpoint, checkpoint_path)
    print(f"  ✅ Saved best model to {checkpoint_path}")

print("\n" + "="*60)
print("Training completed!")
print(f"Best validation loss: {best_validation_loss:.4f}")

Starting training for 10 epochs...
Checkpoints will be saved to: /content/drive/MyDrive/Image-Captioning-System/checkpoints

Epoch 1/10
----------------------------------------


validation: 100%|██████████| 1012/1012 [02:48<00:00,  6.02batches/s, loss=0.1388]



Epoch 1 Results:
  Train Loss: 0.6372
  Val Loss: 0.0889
  ✅ Saved best model to /content/drive/MyDrive/Image-Captioning-System/checkpoints/best_model_epoch_1.pth

Epoch 2/10
----------------------------------------


validation: 100%|██████████| 1012/1012 [02:44<00:00,  6.15batches/s, loss=0.0168]



Epoch 2 Results:
  Train Loss: 0.0692
  Val Loss: 0.0099
  ✅ Saved best model to /content/drive/MyDrive/Image-Captioning-System/checkpoints/best_model_epoch_2.pth

Epoch 3/10
----------------------------------------


...

validation: 100%|██████████| 1012/1012 [02:42<00:00,  6.23batches/s, loss=0.0000]



Epoch 7 Results:
  Train Loss: 0.0001
  Val Loss: 0.0001
  ✅ Saved best model to /content/drive/MyDrive/Image-Captioning-System/checkpoints/best_model_epoch_7.pth

Epoch 8/10
----------------------------------------


validation: 100%|██████████| 1012/1012 [02:46<00:00,  6.07batches/s, loss=0.0000]



Epoch 8 Results:
  Train Loss: 0.0000
  Val Loss: 0.0000
  ✅ Saved best model to /content/drive/MyDrive/Image-Captioning-System/checkpoints/best_model_epoch_8.pth

Epoch 9/10
----------------------------------------


validation:  16%|█▌        | 158/1012 [00:26<01:56,  7.35batches/s, loss=0.0000]